Demand and Welfare
==================

**Author:** Ethan Ligon



Where household surveys earn their keep for policy: what happens to whom
when a price moves.   Also: estimated demand functions can help solve a variety of measurement issues.



## Introduction



### Reading



Deaton is in your `reading/` folder on the hub, or [PDF](https://documents.worldbank.org/curated/en/203811547671768139/pdf/133790-PUB.pdf).

-   Deaton, chs. 4 (§4.2 on Engel curves) and 5 (all)
-   Deaton & Muellbauer (1980), "An Almost Ideal Demand System," *AER* 70:312–326
-   Lewbel (1991), "The rank of demand systems," *Econometrica* 59:711–730
-   Ligon, "Estimating Household Welfare from Disaggregate Expenditures"
    (CUDARE working paper, 2020) — the CFE system we estimate today.  In
    your `reading/` folder on the hub, or
    [eScholarship](https://escholarship.org/uc/item/3ts0g5tn).



## Engel's Laws and Curves



### Ernst Engel



Engel may be the first to use household data in statistical analysis, and used household budget data from Belgium and Saxony to propound what is sometimes called "Engel's (first) Law":
> "The greater the income, the smaller the relative percentage of outlay for subsistence."



### Budget shares and Engel curves



Let $s_{ij} = p_j q_{ij} / x_i$ be household $i$'s budget share on good $j$, $x_i$ total expenditure.  The Working–Leser form,
$$ s_{ij} = \alpha_j + \beta_j \log x_i + \gamma_j' z_i + u_{ij}, $$
fits food shares remarkably well across countries and decades.

-   Homothetic utility implies $\beta_{\text{food}}=0$.
-   $\beta_{\text{food}} < 0$ is Engel's law
-   $z_i$: household size and composition, region, season
-   Estimated by OLS, clustered on the EA



### Estimating the Working–Leser form



Ghana's GLSS7 has both halves of the denominator: food purchases and non-food.  Two recall windows, so put both on a per-year basis before adding them.  (This is **one** way to do it, and probably not the best).



#### Boring Preface



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain
# Figures inside the notebook, whatever the kernel's default backend is.
%matplotlib inline

# The library audits its own corpus on first read and reports what it finds
# --- implausible quantities, NaN index keys, a column that is wholly null in
# one wave --- at multi-paragraph length.  Those reports are a work queue for
# whoever maintains the data, not something the room can act on, and they bury
# the output they are attached to.  Silenced here by their own
# "Set LSMS_..._STRICT=1" signature, which is precise: every other warning,
# pandas deprecations included, still shows.  Delete these two lines to read
# them.
import warnings
warnings.filterwarnings("ignore",
                        message=r"(?s).*Set LSMS_[A-Z_]+=1 to make this fatal")

# The first call that builds a table may print "DVC unavailable ...
# falling back to manual aggregation".  That is about how the library
# fetches its raw files, not about your data; the numbers are the same,
# and it does not recur once the table is built.

import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'lines.linewidth': 1.8})

import lsms_library as ll
import numpy as np, pandas as pd

#### Pull together some data from GLSS7 (2016-17)



In [1]:
ghana = ll.Country('GhanaLSS')
wave = '2016-17'

# The sample table carries the enumeration area v, the weights and the
# rural flag, indexed (i, t).  Load each table once; later cells reuse them.
sample = ghana.sample()

# market='Region' puts a market level m into the index (and takes v out),
# so the same objects serve the demand-system work later in the session.
# waves=[wave] because deriving all seven waves takes three times as long
# and nothing below needs them.  GLSS7 asks food at six visits five days
# apart, so summed over items this is thirty days of purchases.
x_f = ghana.food_expenditures(waves=[wave], market='Region')
x_f = x_f.xs(wave, level='t').squeeze().groupby(['i', 'm']).sum()

# Non-food: Section 9A, one row per item a household reported.  The recall
# window is declared on every row; check it before scaling.
x_n = ghana.nonfood_expenditures(waves=[wave], market='Region').xs(wave, level='t')
print(x_n.RecallWindow.unique())
x_n.head()

#### Scale some things to improve comparability



In [1]:
# One number per household on each side, both per year.
xbar = pd.concat([(x_f * 365.25 / 30).rename('food'),
                  x_n.Expenditure.groupby(['i', 'm']).sum().rename('nonfood')],
                 axis=1)

# A household with food but no 9A row at all did not get the last visit:
# missing, not zero.  Count them, then drop them.
print(f"{xbar.nonfood.isna().sum()} households with food but no Section 9A")
xbar = xbar.dropna()
xbar['total'] = xbar.food + xbar.nonfood
xbar['s'] = xbar.food / xbar.total          # the food share
xbar.describe().round(2)

#### Get household characteristics $z$



The characteristics $z$ are the same as those we'll use later in this
session: children and adults by sex, log household size, and a rural
dummy.  `household_characteristics` counts people by age–sex cell, and
`age_cuts` chooses the cells — one cut at 19 gives children and adults;
the sample table carries the rural flag.



In [1]:
chars = (ghana.household_characteristics(waves=[wave], market='Region', age_cuts=(19,))
              .xs(wave, level='t').groupby(['i', 'm']).first())
s1 = sample.xs(wave, level='t')                  # indexed by i alone
z = chars.join((s1.Rural == 'Rural').astype(float), on='i')

d = pd.concat([xbar.s, np.log(xbar.total).rename('logx'), z], axis=1).astype(float)
d = d.join(s1.v, on='i').dropna()
d = d[np.isfinite(d.logx)]
len(d)

#### Estimate Working-Leser



Now the equation on the slide:
$$ s_{ij} = \alpha_j + \beta_j \log x_i + \gamma_j' z_i + u_{ij}. $$
Use simple OLS to estimate the coefficients and their covariance matrix.
No sampling weights in the regression; they come back when we want population statistics of what it produces.



In [1]:
from metrics_miscellany.estimators import ols

zcols = list(z.columns)
X = d[['logx'] + zcols].assign(const=1.0)
b, V = ols(X, d.s)

coef = b.Coefficients
tab = pd.DataFrame({'coef': coef, 'se': np.sqrt(np.diag(V))})
tab['t'] = tab.coef / tab.se
e = d.s - X @ coef
r2 = 1 - (e**2).sum() / ((d.s - d.s.mean())**2).sum()
print(tab.round(3).to_string())
print(f"\nR^2 = {r2:.3f};  a doubling of x moves the food share by "
      f"{coef['logx'] * np.log(2):+.3f}")

Two things to read off the table.

1.  $\beta_{\text{food}}$ is negative and well over ten standard errors from zero: **Illustrating Engel's law**
2.  This rejects homotheticity (which implies $\beta_{\text{food}}=0$), in one number: a doubling of $x$ lowers the food share by about three points.

Is the line straight?  That's what's needed for the Working-Leser or AIDS demand specifications.

Plot the households, the mean share within each expenditure ventile, and the fitted equation with $z$ held at its mean.



In [1]:
import matplotlib.pyplot as plt

d['bin'] = pd.qcut(d.logx, 20, labels=False, duplicates='drop')
curve = d.s.groupby(d.bin).mean()
mid = d.logx.groupby(d.bin).mean()

zbar = d[zcols].astype(float).mean()
grid = np.linspace(d.logx.quantile(.005), d.logx.quantile(.995), 50)
line = coef['const'] + coef['logx'] * grid + zbar @ coef[zcols]

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(d.logx, d.s, s=3, alpha=0.12, color='0.4', label='households')
ax.plot(mid, curve, 'o', ms=6, label='mean, by ventile')
ax.plot(grid, line, '-', label=r'Working--Leser fit at $\bar z$')
ax.set_xlabel(r'$\log x$  (annual expenditure, cedis)')
ax.set_ylabel('food share of expenditure')
ax.set_ylim(0, 1)
ax.legend(frameon=False, loc='lower left')
plt.show()

The ventile means do not follow the line.  They sit at 0.75–0.78 across
the bottom three-fifths of households and fall only at the top; the
$R^2$ is 0.05.  Engel's law is there, but the "remarkable fit" of the
slide is not, and the aside on Engel's law as a diagnostic says what a
flat share means: something is wrong with the aggregate.  Here we can guess
what.  Section 9A is the *less* frequently purchased non-food.  Soap,
fuel and transport are asked on the food grid and are not in this table,
and rent, utilities, schooling and health are in other sections again.
Three-quarters of what a household spends is not food; three-quarters of
what we can see is.  The `total` understates expenditure and `s` overstates
the share, and it does so most for the households whose non-food is
mostly the frequent kind (the poor) which is what flattens the curve.  
The exercises ask what a fuller denominator does to $\beta_{\text{food}}$.



### Engel's Second Law



A sort of corollary to the first law, sometimes translated as
> "The proportion of the outgo used for food, other things being equal, is the best measure of the material standard of living of a population."

As a welfare measure, this solves some problems that the consumption aggregate has.  In particular:

-   Is a dimensionless share, so don't need to worry at all about price level, etc.
-   If household size/scale can be expressed using Adult Equivalents $A_i^\theta$, then those cancel also.
-   Appears to be naturally comparable across time or place.



### Engel's law across countries



## Other Important Elements of Demand Estimation



### Equivalence scales from behaviour



### Prices, and where to get them (or not)



### Use the market comparison



### Stripping quality out of unit values



Back in Ghana for a moment, with the objects the Working–Leser cell
built.  Unit values for the most-reported food item, and Deaton's
within-cluster regression of their log on $\log x$:
$$ \log v_{ij}^c = a_j^c  + \underbrace{\psi_j \log x_i}_{\mbox{quality}} + \epsilon_{ij}. $$



In [1]:
acq = ghana.food_acquired(waves=[wave]).xs(wave, level='t')
uv = np.log((acq.Expenditure / acq.Quantity).replace([np.inf, -np.inf], np.nan)).dropna()

j0 = uv.groupby('j').size().idxmax()
u0 = uv.xs(j0, level='j').groupby('i').mean().rename('logv')

e = pd.concat([u0, d.droplevel('m')[['logx', 'v']]], axis=1).dropna()

# There are ~1000 clusters, so do not build the dummy matrix; by
# Frisch-Waugh-Lovell, demeaning within cluster gives the identical
# coefficient at a fraction of the cost.
for col in ('logv', 'logx'):
    e[col + '_d'] = e[col].astype(float) - e.groupby('v')[col].transform('mean')

b, V = ols(e[['logx_d']], e.logv_d)
print(f"{j0}: quality elasticity psi = {b.Coefficients['logx_d']:.3f}"
      f"   (se {np.sqrt(V.loc['logx_d', 'logx_d']):.3f})")

A $\psi$ near zero says households of different means pay the same for
this item — it is a homogeneous good.  A large $\psi$ says "onion" in
the questionnaire is several different goods in the market.  Notice that $\psi$ is an *elasticity*, so if we have $\psi=0.36$ we'd interpret that as evidence that 36% of variation in unit prices was due to different quality choices being made by households with different budgets.



### The rank of a demand system



## Constant Frisch Elasticity Demands



### Constant Frisch Elasticity demands



### The estimating equation



Take logs of expenditure on good $j$.  Writing $w_i \equiv -\log \lambda_i$,
$$ \log x^j_{it} \;=\; a^j_{t} \;+\; \beta_j\, w_{it} \;+\; \gamma_j' d_{it} \;+\; \varepsilon^j_{it}. $$

This is a **one-factor model**.  The $(it) \times j$ matrix of log
expenditures, once good-time effects $a^j_t$ and characteristics
$d_{it}$ are swept out, is rank one: loadings $\beta$, factor $w$.

-   estimate by SVD of the residual matrix, with missing data
-   $d_{it}$ enters as good-specific *Barten scales*, not "independence of base" adult equivalents.
-   needs **no prices** and **no total expenditure** — only expenditures on a
    subset of goods



### Estimating the CFE demand system



The system needs several waves and one food classification that holds
across them, coarse enough that most households report most of the goods.
Ghana's GLSS has the waves, and since `lsms_library` 0.14.0 it has an
`Aggregate Label` as well — but that column folds 210 served labels into
204, which is a renaming rather than an aggregation, and a good only a
handful of households ever buy cannot carry a coefficient.  Uganda folds
130 into 76.  So the session switches countries here, once.  The calls are the ones you have already seen, with
`age_cuts=(19,)` for the characteristics.  The system wants a market index
$m$, and we give it a single national market: the estimator's memory
grows with the number of market–wave–good effects it has to sweep out,
and the hub gives each of us three gigabytes.  Exercise 4 asks what four
regional markets change.



In [1]:
from cfe import Regression

# Ghana is done with; the tables it left behind are not small.
del acq, uv, e, x_f, x_n, sample, chars, d

uga = ll.Country('Uganda')
x = uga.food_expenditures(labels='Aggregate').squeeze()
x = x.groupby(['i', 't', 'j']).sum()                   # over sources and EAs
y = np.log(x.replace(0, np.nan).dropna())
y = pd.concat({1: y}, names=['m']).reorder_levels(['i', 't', 'm', 'j']).sort_index()

z = uga.household_characteristics(age_cuts=(19,)).groupby(['i', 't']).first()
z = pd.concat({1: z}, names=['m']).reorder_levels(['i', 't', 'm']).sort_index()

r = Regression(y=y, d=z)
beta = r.get_beta()                                    # a few seconds
print(len(beta), 'goods estimated')

We did not estimate demands for all 76 goods: items too few households
ever report cannot support a coefficient, and the estimator drops them
without being asked.



In [1]:
print("least elastic:"); print(beta.sort_values().head(6).round(2).to_string())
print("\nmost elastic:"); print(beta.sort_values().tail(6).round(2).to_string())
ax = r.graph_beta(xlabel=r'Frisch elasticities $\beta_j$')

Read those two lists before reading anything else.  The ordering is the
model's main testable implication and it is not imposed: nothing in the
estimator knows which goods are staples.  If salt and cassava do not come
out at the bottom and fruit and milk at the top, something is wrong with
the data or with the aggregation.

Household composition enters through $\gamma$, one coefficient per good
per characteristic — good-specific Barten scales rather than a single
equivalence scale imposed on everything:



In [1]:
r.get_gamma().round(2).head(8)

Before trusting any of it, look at the fit, drawn the way the
Working–Leser figure was: a sample of the observations, the mean of the
actual within fifty bins of the predicted, and the line they should lie
on.  This is also the cell that estimates $w$ — the prediction needs
it — so it takes most of a minute.



In [1]:
fit = pd.DataFrame({'actual': r.y,
                    'predicted': r.get_predicted_log_expenditures(fill_missing=False)}).dropna()
fit['bin'] = pd.qcut(fit.predicted, 50, labels=False, duplicates='drop')
mid, mean = fit.groupby('bin').predicted.mean(), fit.groupby('bin').actual.mean()
pts = fit.sample(10000, random_state=0)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts.predicted, pts.actual, s=2, alpha=0.1, color='0.4', label='10,000 observations')
ax.plot(mid, mean, 'o', ms=4, label='mean actual, by bin of predicted')
lim = [fit.predicted.quantile(.001), fit.predicted.quantile(.999)]
ax.plot(lim, lim, 'k--', lw=1.2)
ax.set_xlabel('predicted log expenditure'); ax.set_ylabel('actual')
ax.set_title(f"R^2 = {fit.corr().loc['actual', 'predicted'] ** 2:.2f}")
ax.legend(frameon=False, loc='upper left')
plt.show()

### The Engel pie



An Engel curve shows one good at a time.  With the whole system estimated
we can show all of them at once: the predicted budget *composition* as a
pie, with the radius growing in $\log x$.  Poor households at the
centre, rich at the rim.  Under homotheticity every ring would look the
same; under Engel's law the staples pinch and the fruit widens.



In [1]:
from matplotlib import cm

xhat = r.predicted_expenditures()
xbar = xhat.groupby(['i', 't', 'm']).sum()
p_j = ((r.y.unstack('j') > 0) + 0.).mean()      # prob. good j is bought

lo, hi = xbar.quantile(0.05), xbar.quantile(0.95)
Y = np.geomspace(lo, hi, 60)

fig, ax = plt.subplots(figsize=(7.5, 6))
wedges = None
for k in range(len(Y) - 1, 0, -1):
    ax.set_prop_cycle('color', cm.tab20.colors)
    shares = r.expenditures(Y[k]) * p_j
    radius = 0.3 + 0.7 * np.log(Y[k] / lo) / np.log(hi / lo)
    out = ax.pie(shares, radius=radius, counterclock=False)
    if wedges is None:                      # keep the outermost ring
        wedges, goods = out[0], shares.index.tolist()

big = np.argsort(r.expenditures(Y[-1]).to_numpy() * p_j.to_numpy())[::-1][:10]
ax.legend([wedges[b] for b in big], [goods[b] for b in big],
          loc='center left', bbox_to_anchor=(1.02, 0.5),
          frameon=False, fontsize=8, title='largest ten, at the rim')
plt.show()

Wedges that widen outward are luxuries; wedges that pinch are
necessities.  This is Engel's law for every good at once, and it is the
$\beta$ plot in a form you can show to someone who has never heard of a
Frisch elasticity.



### $w = -\log\lambda$ as a welfare measure



The marginal utility of expenditure $\lambda$ falls as a household
gets better off. Hence $w = -\log\lambda$ increases with welfare.

-   Comparable across households facing different prices — the prices are
    in $a^j_t$, not in $w$
-   Comparable across household compositions — composition is in $\gamma_j' d$
-   Derived from behaviour, not from an assumed equivalence scale or an
    assumed price index



### Welfare comparisons



Two more things come out of the fitted system.  The good–time effects
$a^j_{t}$ contain the prices, and `get_pi` separates them into a price
index $\pi_{t}$ — one number per wave, from expenditures alone —
and relative prices.  Then `get_w` gives $w = -\log\lambda$ for every
household in every wave.



In [1]:
pi = r.get_pi()
pi.droplevel('m').round(2)

Between 2005-06 and 2009-10 the index jumps by about half a log point:
the 2008 food price crisis, found by the demand system with no CPI in
sight.



In [1]:
w = r.get_w()
w.groupby('t').agg(['size', 'mean', 'std']).round(3)

To compare distributions across waves we need a poverty line in units of
$w$.  Pin it the way the World Bank's own figures are pinned: PIP puts
Uganda's 2005 headcount at 58% (USD 2.15 a day, 2017 PPP), so take the 58th
percentile of $w$ in 2005-06 as the line and carry it forward — $w$
is comparable across waves because the prices are in $\pi$ and $a$,
not in it.



In [1]:
from scipy.stats import gaussian_kde

P0 = 0.58                                    # PIP headcount, Uganda 2005
zw = w.xs('2005-06', level='t').quantile(P0)
head = w.groupby('t').apply(lambda s: (s < zw).mean())

waves = sorted(w.index.get_level_values('t').unique())
grid = np.linspace(w.quantile(.005), w.quantile(.995), 300)
fig, ax = plt.subplots(figsize=(6.5, 5))
for k, t in enumerate(waves):
    dens = gaussian_kde(w.xs(t, level='t').dropna())(grid)
    off = (len(waves) - 1 - k) * 0.55
    ax.fill_between(grid, off, off + dens, color=cm.viridis(k / len(waves)), alpha=0.7)
    ax.plot(grid, off + dens, color='k', lw=0.6)
    ax.text(grid[0], off + 0.05, f'{t}  ({head[t]:.0%})', fontsize=9, va='bottom')
ax.axvline(zw, color='r', lw=1)
ax.set_yticks([]); ax.set_xlabel(r'$w = -\log\lambda$')
ax.spines[['left', 'top', 'right']].set_visible(False)
plt.show()

The whole distribution shifts left into 2009-10 and the headcount rises
with it; it recovers by 2010-11, is lowest in 2013-14, and drifts back up
after — the figures are on the plot.  Nominal expenditure rises in every one of these
waves.  That comparison, and what it takes to make it honestly, is
Thursday's.

Does $w$ mean anything to the households themselves?  The Uganda survey asks how
many months in the last twelve the household could not meet its food
needs — self-reported, discrete, and about as far from a demand system
as a welfare question gets.  `months_food_inadequate` carries it for
2009-10 to 2015-16 (in 2009-10, 45% of households report at least one
such month; in the later waves about 20%).  Put the two measures we have
beside it, in the last wave that asks the question.  Expenditure goes in
per adult equivalent on the Oxford scale — 1 for the first adult, 0.7
for each further member aged 14 or more, 0.5 for each child under 14, an
assumed scale as session 3 said it would be — and $w$ goes in as it
is, because composition is already inside it.



In [1]:
mfi = uga.months_food_inadequate()                     # indexed (i, t, v)
hc = uga.household_characteristics(age_cuts=(14,)).groupby(['i', 't']).first()
n14, kids = hc[['F 14+', 'M 14+']].sum(axis=1), hc[['F 00-13', 'M 00-13']].sum(axis=1)
A = 1 + 0.7 * (n14 - 1).clip(lower=0) + 0.5 * kids   # Oxford adult equivalents
q = pd.concat([w.droplevel('m').rename('w'),
               np.log(xbar.droplevel('m') / A).rename('logc')], axis=1)
q = q.reorder_levels(['t', 'i']).join(mfi, how='inner')
q['months'] = pd.cut(q.MonthsInadequate.astype(float),
                     [-0.5, 0.5, 2.5, 5.5, 12.5], labels=['0', '1-2', '3-5', '6+'])
qt = q.xs('2009-10', level='t').dropna(subset=['months', 'logc', 'w'])
cats = qt.months.cat.categories

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, col, lab in zip(axes, ['logc', 'w'],
                        [r'$\log$ food expenditure per adult equivalent', r'$w = -\log\lambda$']):
    data = [qt.loc[qt.months == c, col].to_numpy() for c in cats]
    ax.boxplot(data, tick_labels=[f"{c}\n(n={len(v)})" for c, v in zip(cats, data)],
               showfliers=False, widths=0.6)
    ax.set_xlabel('months without enough food, last 12'); ax.set_ylabel(lab)
fig.suptitle('Uganda 2009-10: two welfare measures against what households report')
plt.show()
qt.groupby('months', observed=True)[['logc', 'w']].median().round(2)

Both measures tend to fall as the reported months rise.  A
self-reported measure is a weak instrument, but on this one the two
agree with it, and with each other, about who is worse off.



## Price Changes



### Welfare consequences of a price change



How much money makes household $i$ as well off at prices $p^1$ as it
was at $p^0$?  With an expenditure function $e(u, p)$,
$$ \mathrm{CV}_i \;=\; e(u^0_i, p^1) - e(u^0_i, p^0), \qquad u^0_i = v(x_i, p^0). $$



### A first-order welfare approximation



For a small change in the price of one good $j$, compensating variation is approximately
$$ \mathrm{CV}_i \;\approx\; q_{ij}\, \Delta p_j \;=\; s_{ij}\, x_i \frac{\Delta p_j}{p_j}. $$

-   to first order you need only the *budget share* — no elasticities
-   the distributional incidence follows from how $s_{ij}$ varies with
    $x_i$, which is the Engel curve you already estimated
-   the exact figure needs the demand system, and says how far the
    approximation can be trusted



### Compensating variation, exactly



The fitted system has an expenditure function, so the exact figure is two
calls: indirect utility at the old prices, expenditure at the new ones.
Prices come from the system itself — $\pi_{t} + A_{tj}$ is the log
price of good $j$ in wave $t$ — so the experiment is the one Uganda
actually ran: 2005-06 prices to 2009-10 prices, for a household with the
median 2005-06 food budget and average characteristics.



In [1]:
logp = r.get_pi() + r.Ar                     # (t, m, j): log prices
p05 = np.exp(logp.xs('2005-06', level='t').groupby('j').mean())
p09 = np.exp(logp.xs('2009-10', level='t').groupby('j').mean())

x0 = xbar.xs('2005-06', level='t').median()          # median food budget
U0 = r.indirect_utility(x0, p05)
cv = lambda p1: float(r.expenditure(U0, p1)) - x0     # e(U0, p1) - e(U0, p05)

print(f"median 2005-06 food budget: {x0:,.0f} shillings")
print(f"CV of the 2005-06 -> 2009-10 price changes: {cv(p09):,.0f} shillings"
      f"  ({cv(p09) / x0:.0%} of the budget)")

Two-thirds of the budget: that is what the crisis cost the median
household, in the money of the day.  Now one good.  Matoke has a large
Frisch elasticity and its price rose by about 60% between the two waves;
raise it alone, and compare the exact answer with the first-order one on
the slide.



In [1]:
j = 'Matoke'
s_j = r.expenditures(x0, p05)[j] / x0          # the share the system gives at x0
k_obs = p09[j] / p05[j]
mult = np.linspace(0.5, 3, 26)
exact = np.array([cv(p05.where(p05.index != j, p05[j] * k)) for k in mult])
first = s_j * x0 * (mult - 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mult, exact / x0, label='exact: $e(u^0, p^1) - e(u^0, p^0)$')
ax.plot(mult, first / x0, '--', label=r'first order: $s_j\,x\,\Delta p_j / p_j$')
ax.axvline(k_obs, color='0.5', lw=1); ax.axhline(0, color='0.5', lw=1)
ax.set_xlabel(f'price of {j}, relative to 2005-06')
ax.set_ylabel('compensating variation / budget')
ax.legend(frameon=False)
plt.show()

e_obs = cv(p05.where(p05.index != j, p09[j])); f_obs = s_j * x0 * (k_obs - 1)
print(f"at the observed change (x{k_obs:.2f}): exact {e_obs:,.0f}, "
      f"first order {f_obs:,.0f}; ratio {e_obs / f_obs:.2f}")

The first-order line is tangent at the old price and overstates the cost
everywhere else, because the household substitutes away and the
approximation does not know it.  At the change Uganda saw, the
overstatement is about a fifth, and it grows with the change.  How much that
matters depends on the question, which is what the aside is about.



## Exercises



### Exercise 1: compare Engel curves



The Working–Leser cell uses Section 9A alone in its non-food denominator.

1.  Interact `logx` with `Rural`. Does the slope differ between urban
    and rural households?
2.  Re-estimate on the 2012-13 wave, which has the same section.
    Has $\beta$ moved in five years?



### Exercise 1: compare expenditure coverage



Uganda's `nonfood_expenditures()` includes frequently and infrequently
purchased non-food items. Put them on a common recall basis:

-   The questionnaire specifies 30 days for non-durables and 365 for
    semi-durables in 2019-20 (see session 3).
-   Use `uga.categorical_mapping['nonfood_items']` to identify items.
-   Repeat the estimation for Uganda 2019-20. Is the share still flat
    over most of the distribution? How do the composition coefficients change?



### Exercise 1: account for clustered sampling



The reported standard errors allow heteroskedasticity but don't account
for dependence within an EA.

-   Cluster on the EA, using `d.v`.
-   By how much do the standard errors grow?
-   Is that the $\sqrt{\text{deff}}$ of session 1?



### Exercise 2: equivalence scales



Implement the Engel equivalence scale.

Find the expenditure ratio that equates the predicted food share of:

-   a household with two adults and two children;
-   a household with two adults.

Compare with the $\theta$-scale from session 2.



### Exercise 3: who bears the price change?



For the 2005-06 to 2009-10 price change:

1.  Compute the first-order welfare cost at each decile of the 2005-06
    food budget, using predicted budget shares.
2.  Compute exact compensating variation at the same deciles.
3.  Who loses most in shillings? Who loses most as a share of the budget?
    Are those the same households?



### Exercise 4: define regional markets



Re-estimate the CFE system with the four regions as markets.
Use `market='Region'` on both tables instead of assigning every
household to one national market.

-   How much do the $\beta_j$ move?
-   What does $\pi$ look like by region?
-   What did one national market assume away?

Watch your kernel's memory while you do it.



### Exercise 5: compare welfare measures



Regress $w$ on log total food expenditure.

-   How closely do the two measures agree?
-   For which households do they disagree?
-   What adjustments does the demand-based measure make that expenditure
    alone doesn't?

